Imports

In [35]:
import os
import json
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print("Imports successful!")

Imports successful!


#### V1.1

Load LLM + Embeddings

In [36]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    model="google/gemma-3-27b-it",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LLM + Embeddings ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM + Embeddings ready!


Load Existing FAISS Index

In [37]:
vector_store = FAISS.load_local(
    "../data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Index Loaded!")

FAISS Index Loaded!


Create Retriever

In [38]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 10
    }
)

print("Retriever Ready!")

Retriever Ready!


Retrieve Context

In [39]:
topic = "Machine Learning"

docs = retriever.invoke(topic)

context = "\n\n".join(
    doc.page_content for doc in docs
)

print(context[:1000])

Introduction to Machine Learning
Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn patterns
from data and make predictions without being explicitly programmed.
There are three main types of machine learning:
1. Supervised Learning - Uses labeled data. Examples: Email spam detection, house price
prediction.
2. Unsupervised Learning - Uses unlabeled data. Examples: Customer segmentation, clustering.

Generative AI and Large Language Models
Generative AI focuses on creating new content such as text, images, audio, and code.
Large Language Models (LLMs) are trained on massive datasets and can generate human-like text.
Examples: GPT, Claude, Gemini, Llama.
Important Concepts:
• Prompt Engineering
• Retrieval Augmented Generation (RAG)
• Fine-Tuning
• AI Agents
RAG combines information retrieval with language generation to provide accurate and
context-aware responses.

Deep Learning and Neural Networks
Deep Learning is a subset of Machine Learning t

Generate Quiz

In [40]:
prompt = f"""
You are a quiz generator.

Generate 5 MCQs from the context.

IMPORTANT:
- Return ONLY valid JSON.
- Do NOT use markdown.
- Do NOT wrap output in ```json.
- Do NOT add explanations.
- Output must start with [ and end with ].

Context:
{context}
"""

response = llm.invoke(prompt)

quiz_json = response.content

print(quiz_json)

[
  {
    "question": "What is the primary characteristic of Supervised Learning?",
    "options": [
      "It uses unlabeled data.",
      "It learns through rewards and penalties.",
      "It uses labeled data.",
      "It focuses on creating new content."
    ],
    "answer": "It uses labeled data."
  },
  {
    "question": "Which of the following is an example of Unsupervised Learning?",
    "options": [
      "Email spam detection",
      "House price prediction",
      "Customer segmentation",
      "Game-playing AI"
    ],
    "answer": "Customer segmentation"
  },
  {
    "question": "What does Generative AI primarily focus on?",
    "options": [
      "Analyzing existing data",
      "Creating new content",
      "Predicting future outcomes",
      "Automating repetitive tasks"
    ],
    "answer": "Creating new content"
  },
  {
    "question": "What is Retrieval Augmented Generation (RAG)?",
    "options": [
      "A type of supervised learning algorithm.",
      "A method f

Parse JSON

In [41]:
import json
import re

clean_json = re.sub(r"```json|```", "", quiz_json).strip()

quiz = json.loads(clean_json)

print(f"Total Questions: {len(quiz)}")

Total Questions: 5


Display Quiz

In [42]:
for i, q in enumerate(quiz, start=1):
    print(f"\nQ{i}. {q['question']}")

    for option in q["options"]:
        print(f"  • {option}")

    print(f"✅ Answer: {q['answer']}")


Q1. What is the primary characteristic of Supervised Learning?
  • It uses unlabeled data.
  • It learns through rewards and penalties.
  • It uses labeled data.
  • It focuses on creating new content.
✅ Answer: It uses labeled data.

Q2. Which of the following is an example of Unsupervised Learning?
  • Email spam detection
  • House price prediction
  • Customer segmentation
  • Game-playing AI
✅ Answer: Customer segmentation

Q3. What does Generative AI primarily focus on?
  • Analyzing existing data
  • Creating new content
  • Predicting future outcomes
  • Automating repetitive tasks
✅ Answer: Creating new content

Q4. What is Retrieval Augmented Generation (RAG)?
  • A type of supervised learning algorithm.
  • A method for creating new datasets.
  • Combining information retrieval with language generation.
  • A deep learning framework.
✅ Answer: Combining information retrieval with language generation.

Q5. Deep Learning is a subset of which field?
  • Artificial Intelligence

Scoring System

In [44]:
score = 0

for i, q in enumerate(quiz, start=1):

    print(f"\nQ{i}. {q['question']}")

    for idx, option in enumerate(q["options"], start=1):
        print(f"{idx}. {option}")

    user_choice = int(input("Enter option number: "))

    selected = q["options"][user_choice - 1]

    if selected == q["answer"]:
        print("✅ Correct")
        score += 1
    else:
        print(f"❌ Wrong. Correct Answer: {q['answer']}")

print(f"\nFinal Score: {score}/{len(quiz)}")


Q1. What is the primary characteristic of Supervised Learning?
1. It uses unlabeled data.
2. It learns through rewards and penalties.
3. It uses labeled data.
4. It focuses on creating new content.
❌ Wrong. Correct Answer: It uses labeled data.

Q2. Which of the following is an example of Unsupervised Learning?
1. Email spam detection
2. House price prediction
3. Customer segmentation
4. Game-playing AI
✅ Correct

Q3. What does Generative AI primarily focus on?
1. Analyzing existing data
2. Creating new content
3. Predicting future outcomes
4. Automating repetitive tasks
✅ Correct

Q4. What is Retrieval Augmented Generation (RAG)?
1. A type of supervised learning algorithm.
2. A method for creating new datasets.
3. Combining information retrieval with language generation.
4. A deep learning framework.
✅ Correct

Q5. Deep Learning is a subset of which field?
1. Artificial Intelligence
2. Natural Language Processing
3. Cybersecurity
4. Data Mining
✅ Correct

Final Score: 4/5


#### V2.2 Flashcards

Flashcards

In [45]:
prompt = f"""
Using the context below,

Generate 10 flashcards.

Return ONLY JSON.

Format:

[
  {{
    "front": "...",
    "back": "..."
  }}
]

Context:

{context}
"""

response = llm.invoke(prompt)

flashcards_json = response.content

print(flashcards_json)

```json
[
  {
    "front": "What is Machine Learning?",
    "back": "A branch of AI that enables computers to learn from data and make predictions without explicit programming."
  },
  {
    "front": "Name the three main types of Machine Learning.",
    "back": "Supervised Learning, Unsupervised Learning, and Reinforcement Learning."
  },
  {
    "front": "What type of data does Supervised Learning use?",
    "back": "Labeled data."
  },
  {
    "front": "Give an example of Supervised Learning.",
    "back": "Email spam detection or house price prediction."
  },
  {
    "front": "What type of data does Unsupervised Learning use?",
    "back": "Unlabeled data."
  },
  {
    "front": "What does Generative AI focus on?",
    "back": "Creating new content like text, images, audio, and code."
  },
  {
    "front": "Name a few examples of Large Language Models (LLMs).",
    "back": "GPT, Claude, Gemini, Llama."
  },
  {
    "front": "What is Retrieval Augmented Generation (RAG)?",
    "back"

Parse Flashcards

In [46]:
import re
import json

clean_json = re.sub(
    r"```json|```",
    "",
    flashcards_json
).strip()

flashcards = json.loads(clean_json)

print(f"Total Flashcards: {len(flashcards)}")

Total Flashcards: 10


Display Flashcards

In [47]:
for i, card in enumerate(flashcards, start=1):

    print("\n" + "="*50)

    print("Front:")
    print(card["front"])

    print("\nBack:")
    print(card["back"])


Front:
What is Machine Learning?

Back:
A branch of AI that enables computers to learn from data and make predictions without explicit programming.

Front:
Name the three main types of Machine Learning.

Back:
Supervised Learning, Unsupervised Learning, and Reinforcement Learning.

Front:
What type of data does Supervised Learning use?

Back:
Labeled data.

Front:
Give an example of Supervised Learning.

Back:
Email spam detection or house price prediction.

Front:
What type of data does Unsupervised Learning use?

Back:
Unlabeled data.

Front:
What does Generative AI focus on?

Back:
Creating new content like text, images, audio, and code.

Front:
Name a few examples of Large Language Models (LLMs).

Back:
GPT, Claude, Gemini, Llama.

Front:
What is Retrieval Augmented Generation (RAG)?

Back:
Combines information retrieval with language generation for accurate, context-aware responses.

Front:
What is Deep Learning a subset of?

Back:
Machine Learning.

Front:
Name a popular Deep Le

#### V2.3: Progress Tracking

SQLite Setup

In [48]:
import sqlite3

conn = sqlite3.connect("learning.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS quiz_results (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT,
    score INTEGER,
    total INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()

print("Database Ready!")

Database Ready!


Save Quiz Result

In [49]:
score = 4
total = 5
topic = "Machine Learning"

cursor.execute("""
INSERT INTO quiz_results (
    topic,
    score,
    total
)
VALUES (?, ?, ?)
""", (topic, score, total))

conn.commit()

print("Result Saved!")

Result Saved!


View History

In [50]:
cursor.execute("""
SELECT *
FROM quiz_results
ORDER BY created_at DESC
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(5, 'Machine Learning', 4, 5, '2026-06-03 13:33:20')
(3, 'Generative AI', 2, 10, '2026-06-03 13:30:43')
(4, 'Deep Learning', 4, 10, '2026-06-03 13:30:43')
(2, 'Machine Learning', 4, 5, '2026-06-03 13:26:47')
(1, 'Machine Learning', 4, 5, '2026-06-03 13:25:06')


Accuracy %

In [51]:
cursor.execute("""
SELECT
SUM(score),
SUM(total)
FROM quiz_results
""")

correct, attempted = cursor.fetchone()

accuracy = (correct / attempted) * 100

print(f"Overall Accuracy: {accuracy:.2f}%")

Overall Accuracy: 51.43%


#### V2.4 — Weak Topic Detection

Topic Accuracy

In [52]:
cursor.execute("""
SELECT
    topic,
    SUM(score) as correct,
    SUM(total) as attempted,
    ROUND(
        (SUM(score) * 100.0) /
        SUM(total),
        2
    ) as accuracy
FROM quiz_results
GROUP BY topic
ORDER BY accuracy ASC
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('Generative AI', 2, 10, 20.0)
('Deep Learning', 4, 10, 40.0)
('Machine Learning', 12, 15, 80.0)


Detect Weak Topics

In [53]:
WEAK_THRESHOLD = 70

cursor.execute("""
SELECT
    topic,
    ROUND(
        (SUM(score) * 100.0) /
        SUM(total),
        2
    ) as accuracy
FROM quiz_results
GROUP BY topic
""")

rows = cursor.fetchall()

weak_topics = []

for topic, accuracy in rows:

    if accuracy < WEAK_THRESHOLD:
        weak_topics.append((topic, accuracy))

print("Weak Topics:\n")

for topic, accuracy in weak_topics:
    print(f"{topic}: {accuracy}%")

Weak Topics:

Deep Learning: 40.0%
Generative AI: 20.0%


Personalized Recommendation

In [54]:
if not weak_topics:
    print("🎉 No weak topics detected!")

else:

    print("📚 Recommended Revision Topics:\n")

    for topic, accuracy in weak_topics:

        print(
            f"- {topic} "
            f"(Accuracy: {accuracy}%)"
        )

📚 Recommended Revision Topics:

- Deep Learning (Accuracy: 40.0%)
- Generative AI (Accuracy: 20.0%)


#### V2.5 Adaptive Learning Engine

Get Weakest Topic

In [55]:
cursor.execute("""
SELECT
    topic,
    ROUND(
        (SUM(score) * 100.0) /
        SUM(total),
        2
    ) as accuracy
FROM quiz_results
GROUP BY topic
ORDER BY accuracy ASC
LIMIT 1
""")

result = cursor.fetchone()

weakest_topic = result[0]

print(f"Weakest Topic: {weakest_topic}")

Weakest Topic: Generative AI


Retrieve Topic Context

In [56]:
docs = retriever.invoke(weakest_topic)

context = "\n\n".join(
    doc.page_content for doc in docs
)

print(context[:1000])

Generative AI and Large Language Models
Generative AI focuses on creating new content such as text, images, audio, and code.
Large Language Models (LLMs) are trained on massive datasets and can generate human-like text.
Examples: GPT, Claude, Gemini, Llama.
Important Concepts:
• Prompt Engineering
• Retrieval Augmented Generation (RAG)
• Fine-Tuning
• AI Agents
RAG combines information retrieval with language generation to provide accurate and
context-aware responses.

Introduction to Machine Learning
Machine Learning (ML) is a branch of Artificial Intelligence that enables computers to learn patterns
from data and make predictions without being explicitly programmed.
There are three main types of machine learning:
1. Supervised Learning - Uses labeled data. Examples: Email spam detection, house price
prediction.
2. Unsupervised Learning - Uses unlabeled data. Examples: Customer segmentation, clustering.

Deep Learning and Neural Networks
Deep Learning is a subset of Machine Learning t

Generate Revision Quiz

In [59]:
prompt = f"""
Generate 5 MCQs only from the context.

Return valid JSON only.

Context:

{context}
"""

In [60]:
response = llm.invoke(prompt)

revision_quiz = response.content

print(revision_quiz)

```json
[
  {
    "question": "What is the primary focus of Generative AI?",
    "options": [
      "Analyzing existing data",
      "Creating new content",
      "Predicting future outcomes",
      "Automating repetitive tasks"
    ],
    "answer": "Creating new content"
  },
  {
    "question": "Which of the following is an example of a Large Language Model (LLM)?",
    "options": [
      "TensorFlow",
      "Keras",
      "GPT",
      "PyTorch"
    ],
    "answer": "GPT"
  },
  {
    "question": "What does RAG stand for?",
    "options": [
      "Rapid Application Generation",
      "Retrieval Augmented Generation",
      "Recursive Algorithm Guidance",
      "Reinforcement and Gradient Adjustment"
    ],
    "answer": "Retrieval Augmented Generation"
  },
  {
    "question": "Which type of Machine Learning utilizes labeled data?",
    "options": [
      "Unsupervised Learning",
      "Reinforcement Learning",
      "Supervised Learning",
      "Deep Learning"
    ],
    "answer": "

Learning Recommendation

In [61]:
accuracy = result[1]

if accuracy < 50:
    level = "High Priority"

elif accuracy < 80:
    level = "Needs Revision"

else:
    level = "Good"

print(f"""
Topic: {weakest_topic}
Accuracy: {accuracy}%
Status: {level}
""")


Topic: Generative AI
Accuracy: 20.0%
Status: High Priority

